 # Predicting Movie Rental Durations

 A regression modeling and feature engineering project to predict rental duration,

 helping a DVD rental company optimize their inventory management.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 1. LOAD THE DATA
# ---------------------------------------------------------
# Loading the dataset containing historical rental information.
df = pd.read_csv("https://raw.githubusercontent.com/amdjadouxx/Predicting-Movie-Rental-Durations/main/rental_info.csv")
df.head()


In [ ]:
# ---------------------------------------------------------
# 2. FEATURE ENGINEERING: TARGET VARIABLE
# ---------------------------------------------------------
# The objective is to predict the number of days a movie is rented.
# We create our target variable 'rental_length_days' by subtracting the 
# rental date from the return date.
df["rental_date"] = pd.to_datetime(df["rental_date"])
df["return_date"] = pd.to_datetime(df["return_date"])
df["rental_length_days"] = (df["return_date"] - df["rental_date"]).dt.days


In [ ]:
# ---------------------------------------------------------
# 3. FEATURE ENGINEERING: DUMMY VARIABLES
# ---------------------------------------------------------
# The 'special_features' column contains text describing extra content.
# We extract 'Deleted Scenes' and 'Behind the Scenes' as binary indicators (1 or 0)
# since these features might influence how long a customer keeps a DVD.
df["deleted_scenes"] = np.where(df["special_features"].str.contains("Deleted Scenes"), 1, 0)
df["behind_the_scenes"] = np.where(df["special_features"].str.contains("Behind the Scenes"), 1, 0)


In [ ]:
# ---------------------------------------------------------
# 4. DATA SPLITTING
# ---------------------------------------------------------
# Drop columns that leak information about the target variable (dates) 
# or are unstructured text that we already processed (special_features).
drop_cols = ["rental_date", "return_date", "special_features", "rental_length_days"]
X = df.drop(columns=drop_cols)
y = df["rental_length_days"] # Target variable

# Split the data into training (80%) and testing (20%) sets.
# random_state ensures reproducibility for the specific DataCamp requirements.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)


In [ ]:
# ---------------------------------------------------------
# 5. FEATURE SELECTION WITH LASSO REGRESSION
# ---------------------------------------------------------
# Lasso regression requires scaled features. We fit the scaler on the training data
# and transform both training and testing data to prevent data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply Lasso (L1 regularization) to shrink less important feature coefficients to exactly 0.
lasso = Lasso(alpha=0.3, random_state=9)
lasso.fit(X_train_scaled, y_train)

# Select only the features where the Lasso coefficient is greater than 0.
selected_features = X.columns[lasso.coef_ > 0]
print("Selected features:", list(selected_features))

# Subset our training and testing data to only include the selected features.
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]


In [ ]:
# ---------------------------------------------------------
# 6. MODEL TRAINING & HYPERPARAMETER TUNING
# ---------------------------------------------------------
# We use a Random Forest Regressor as it handles non-linear relationships well.
rf = RandomForestRegressor(random_state=9)

# Define a grid of hyperparameters to search through.
param_dist = {
    'n_estimators': np.arange(50, 200, 25),
    'max_depth': np.arange(5, 20, 2),
    'min_samples_split': [2, 5, 10]
}

# Perform a randomized search to find the optimal hyperparameters efficiently.
# We use 3-fold cross-validation and optimize for negative mean squared error.
rand_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=9,
    n_jobs=-1
)

rand_search.fit(X_train_selected, y_train)


In [ ]:
# ---------------------------------------------------------
# 7. MODEL EVALUATION
# ---------------------------------------------------------
# Retrieve the best model from the randomized search.
best_model = rand_search.best_estimator_

# Predict rental durations on the unseen test dataset.
y_pred = best_model.predict(X_test_selected)

# Calculate the Mean Squared Error (MSE). 
# The project goal is to achieve an MSE of less than 3.
mse = mean_squared_error(y_test, y_pred)

print(f"Best hyperparameters: {rand_search.best_params_}")
print(f"Test Set Mean Squared Error (MSE): {mse:.4f}")